# 🏥 MediCare Patient Follow-Up Agent
### Agentic AI Prototype — Data Scientist Assessment

**Domain:** Healthcare | **Role:** Data Scientist | **Format:** Jupyter Notebook + CSV

This notebook implements an AI agent that **autonomously orchestrates tool calls** to:
- Analyse 100 patient records
- Flag clinical risks using lab values, vitals, and visit history
- Generate prioritised follow-up care actions

| Task | Description |
|------|-------------|
| Task 1 | Data Loading & Exploratory Analysis |
| Task 2 | Define Agent Tools |
| Task 3 | Implement the Agentic Loop |
| Task 4 | Single Patient Analysis |
| Task 5 | Missed Appointment Follow-Up |

---
## ⚙️ Setup — Install Dependencies

In [ ]:
# Install required libraries (run once)
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'openai', 'pandas',
     'matplotlib', 'seaborn', '--quiet'],
    check=True
)
print('✅ Dependencies ready.')

---
## Task 1 — Data Loading & Exploratory Analysis

Load `patient_data.csv`, inspect structure, and visualise key clinical distributions.

In [ ]:
import os, json, warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

# Load dataset
df = pd.read_csv('patient_data.csv')
print(f'Dataset loaded: {df.shape[0]} patients, {df.shape[1]} columns')
df.head()

In [ ]:
print('=== Column Types ===')
print(df.dtypes.to_string())
print('\n=== Missing Values ===')
print(df.isnull().sum().to_string())
print('\n=== Descriptive Statistics ===')
df[['age', 'days_since_last_visit', 'vitals_bp_systolic',
    'vitals_bp_diastolic', 'vitals_heart_rate', 'vitals_spo2']].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('MediCare Patient Dataset — Exploratory Analysis', fontsize=14, fontweight='bold')

# 1. Age distribution
axes[0,0].hist(df['age'], bins=15, color='steelblue', edgecolor='white')
axes[0,0].set_title('Age Distribution')
axes[0,0].set_xlabel('Age (years)')
axes[0,0].set_ylabel('Count')

# 2. Top diagnoses
diag_counts = df['diagnosis'].value_counts().head(10)
diag_counts.plot(kind='barh', ax=axes[0,1], color='coral')
axes[0,1].set_title('Top 10 Diagnoses')
axes[0,1].invert_yaxis()

# 3. Gender split
df['gender'].value_counts().plot(
    kind='pie', ax=axes[0,2], autopct='%1.0f%%',
    colors=['#5b9bd5', '#ed7d31'], startangle=90
)
axes[0,2].set_title('Gender Split')
axes[0,2].set_ylabel('')

# 4. Systolic BP
axes[1,0].hist(df['vitals_bp_systolic'], bins=15, color='tomato', edgecolor='white')
axes[1,0].axvline(140, color='black', linestyle='--', linewidth=1.5, label='HTN threshold (140)')
axes[1,0].axvline(160, color='darkred', linestyle=':', linewidth=1.5, label='Severe HTN (160)')
axes[1,0].set_title('Systolic BP Distribution')
axes[1,0].legend(fontsize=8)

# 5. Days since last visit
axes[1,1].hist(df['days_since_last_visit'], bins=15, color='mediumseagreen', edgecolor='white')
axes[1,1].axvline(180, color='red', linestyle='--', linewidth=1.5, label='180-day threshold')
axes[1,1].set_title('Days Since Last Visit')
axes[1,1].legend(fontsize=8)

# 6. Missed appointments
missed_counts = df['missed_last_appointment'].value_counts()
missed_counts.plot(kind='bar', ax=axes[1,2], color=['#2ecc71', '#e74c3c'], rot=0)
axes[1,2].set_title('Missed Last Appointment')
for bar, val in zip(axes[1,2].patches, missed_counts):
    axes[1,2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                   str(val), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Patients who missed last appointment: {(df.missed_last_appointment=="Yes").sum()}')
print(f'Patients overdue (>180 days):         {(df.days_since_last_visit > 180).sum()}')
print(f'High systolic BP (>=160):              {(df.vitals_bp_systolic >= 160).sum()}')

---
## Task 2 — Define Agent Tools

Four tools are defined and exposed as **OpenAI function-calling schemas**.
The LLM decides autonomously which tools to invoke and in what order.

| Tool | Description |
|------|-------------|
| `get_patient_data` | Fetch complete clinical record for a patient ID |
| `flag_clinical_risk` | Evaluate lab values & vitals against clinical thresholds → risk level |
| `generate_care_actions` | Produce prioritised follow-up actions from risk flags |
| `get_missed_appointment_patients` | Return all patients who missed their last appointment |

### Clinical Thresholds Used
| Lab Test | Risk Threshold | Condition |
|----------|----------------|-----------|
| HbA1c | > 8.0% | Poor glycaemic control |
| BNP | > 400 pg/mL | Elevated heart failure marker |
| eGFR | < 45 mL/min | Reduced kidney function |
| Hemoglobin | < 10 g/dL | Significant anaemia |
| FEV1% | < 50% | Severe airflow obstruction |
| Peak Flow | < 320 L/min | Reduced respiratory function |
| PHQ-9 | > 14/27 | Moderate-severe depression |
| TSH | > 6.0 mIU/L | Undertreated hypothyroidism |
| Systolic BP (vitals) | ≥ 160 mmHg | Severely elevated BP |
| SpO2 | < 94% | Low oxygen saturation |

### 🔑 API Key Configuration
Set your OpenAI API key as an environment variable, or paste it directly below (mask before submitting).

In [ ]:
from openai import OpenAI

# ── API Key Setup ─────────────────────────────────────────────────────────────
# Recommended: set environment variable
#   Windows:  $env:OPENAI_API_KEY = 'sk-...'
#   macOS:    export OPENAI_API_KEY='sk-...'
# Or paste key directly (replace below — mask 'sk-...' before submission)
API_KEY = os.getenv('OPENAI_API_KEY', 'sk-PASTE-YOUR-KEY-HERE')
MODEL   = 'gpt-4o-mini'   # Switch to 'gpt-4o' for higher accuracy

client = OpenAI(api_key=API_KEY)
print(f'✅ OpenAI client initialised — model: {MODEL}')

In [ ]:
# ─── Tool 1: get_patient_data ─────────────────────────────────────────────────
def get_patient_data(patient_id: str) -> dict:
    """Retrieve complete patient record from the dataset by patient ID."""
    row = df[df['patient_id'] == patient_id]
    if row.empty:
        return {'error': f'Patient {patient_id} not found in dataset'}
    return row.iloc[0].to_dict()


# ─── Tool 2: flag_clinical_risk ───────────────────────────────────────────────
def flag_clinical_risk(patient_id: str) -> dict:
    """Evaluate patient lab values, vitals, and visit history against
    clinical thresholds to determine risk level: LOW | MEDIUM | HIGH."""
    rec = get_patient_data(patient_id)
    if 'error' in rec:
        return rec

    flags = []
    lab, val = rec['lab_test'], float(rec['lab_value'])

    # Lab value thresholds
    lab_rules = {
        'HbA1c':       (val > 8.0,  f'HbA1c {val}% (target <8%) — poor glycaemic control'),
        'BNP':         (val > 400,  f'BNP {val} pg/mL (>400) — elevated heart failure marker'),
        'eGFR':        (val < 45,   f'eGFR {val} mL/min/1.73m² (<45) — reduced kidney function'),
        'Hemoglobin':  (val < 10,   f'Haemoglobin {val} g/dL (<10) — significant anaemia'),
        'FEV1%':       (val < 50,   f'FEV1% {val}% (<50) — severe airflow obstruction'),
        'Peak Flow':   (val < 320,  f'Peak Flow {val} L/min (<320) — reduced respiratory function'),
        'BP Systolic': (val > 140,  f'Lab BP {val} mmHg (>140) — hypertension'),
        'PHQ-9 Score': (val > 14,   f'PHQ-9 {val}/27 (>14) — moderate-to-severe depression'),
        'TSH':         (val > 6.0,  f'TSH {val} mIU/L (>6.0) — hypothyroidism undertreated'),
    }
    if lab in lab_rules and lab_rules[lab][0]:
        flags.append(lab_rules[lab][1])

    # Vitals thresholds
    if int(rec['vitals_bp_systolic']) >= 160:
        flags.append(f"Vitals systolic BP {rec['vitals_bp_systolic']} mmHg — severely elevated")
    if int(rec['vitals_spo2']) < 94:
        flags.append(f"SpO2 {rec['vitals_spo2']}% — low oxygen saturation")

    # Visit & appointment flags
    if int(rec['days_since_last_visit']) > 180:
        flags.append(f"Last visit {rec['days_since_last_visit']} days ago — overdue follow-up")
    if str(rec['missed_last_appointment']) == 'Yes':
        flags.append('Missed last scheduled appointment')

    # Risk stratification
    n = len(flags)
    risk_level = 'HIGH' if n >= 3 else ('MEDIUM' if n >= 1 else 'LOW')

    return {
        'patient_id':   patient_id,
        'patient_name': rec['patient_name'],
        'age':          rec['age'],
        'diagnosis':    rec['diagnosis'],
        'risk_level':   risk_level,
        'flag_count':   n,
        'flags':        flags
    }


# ─── Tool 3: generate_care_actions ────────────────────────────────────────────
def generate_care_actions(patient_id: str, risk_assessment: dict) -> dict:
    """Generate a prioritised list of follow-up care actions for a patient
    based on their risk flags. Returns structured action plan."""
    rec = get_patient_data(patient_id)
    if 'error' in rec:
        return rec

    actions = []
    level = risk_assessment.get('risk_level', 'LOW')
    flags = risk_assessment.get('flags', [])

    # Urgency header
    if level == 'HIGH':
        actions.append('🚨 URGENT: Contact patient within 24 hours')
    elif level == 'MEDIUM':
        actions.append('⚠️  PRIORITY: Schedule review within 1 week')

    # Condition-specific actions
    flag_text = ' '.join(flags).lower()
    if 'hba1c' in flag_text:
        actions.append('Diabetes: Schedule endocrinology review; consider insulin titration or regimen change')
    if 'bnp' in flag_text:
        actions.append('Heart Failure: Cardiology referral; review and adjust diuretic (Furosemide) dosage')
    if 'egfr' in flag_text or 'kidney' in flag_text:
        actions.append('CKD: Nephrology referral; renal diet counselling; monitor electrolytes monthly')
    if 'haemoglobin' in flag_text or 'anaemia' in flag_text:
        actions.append('Anaemia: Review iron supplementation; consider IV iron or transfusion; repeat CBC in 4 weeks')
    if 'fev1' in flag_text or 'airflow' in flag_text:
        actions.append('COPD: Pulmonology review; consider LAMA/LABA step-up; pulmonary rehab referral')
    if 'peak flow' in flag_text or 'respiratory' in flag_text:
        actions.append('Asthma: Update asthma action plan; step-up inhaler therapy; peak flow monitoring diary')
    if 'phq-9' in flag_text or 'depression' in flag_text:
        actions.append('Mental Health: Urgent psychiatric review; suicide risk assessment; crisis resource provision')
    if 'tsh' in flag_text or 'hypothyroid' in flag_text:
        actions.append('Thyroid: Endocrinology review; titrate Levothyroxine dose; repeat TSH in 6 weeks')
    if 'systolic' in flag_text or 'hypertension' in flag_text:
        actions.append('Hypertension: Medication review; consider dose escalation or combination therapy; home BP monitoring')
    if 'spo2' in flag_text or 'oxygen' in flag_text:
        actions.append('Respiratory: Arrange pulse oximetry monitoring; consider supplemental O2 assessment')
    if 'overdue' in flag_text:
        actions.append('Access: Schedule urgent catch-up appointment; identify and address attendance barriers')
    if 'missed' in flag_text:
        actions.append('Adherence: Outreach call/SMS to reschedule missed appointment; explore support needs')

    # Always include medication check
    actions.append(f'Medication: Review adherence to {rec["current_medication"]}; check for side-effects')

    return {
        'patient_id':          patient_id,
        'patient_name':        rec['patient_name'],
        'age':                 rec['age'],
        'diagnosis':           rec['diagnosis'],
        'current_medication':  rec['current_medication'],
        'risk_level':          level,
        'recommended_actions': list(dict.fromkeys(actions))  # deduplicate, preserve order
    }


# ─── Tool 4: get_missed_appointment_patients ──────────────────────────────────
def get_missed_appointment_patients() -> dict:
    """Return the full list of patients who missed their last scheduled appointment."""
    missed = df[df['missed_last_appointment'] == 'Yes'][[
        'patient_id', 'patient_name', 'age', 'diagnosis',
        'days_since_last_visit', 'next_scheduled_visit'
    ]].to_dict(orient='records')
    return {'total_missed': len(missed), 'patients': missed}


# Quick smoke-test
test = flag_clinical_risk('P0008')
print(f'Smoke-test P0008: risk={test["risk_level"]}, flags={test["flag_count"]}')
print('✅ All 4 tools defined and working.')

In [ ]:
# ─── OpenAI Function-Calling Schemas ─────────────────────────────────────────
TOOLS = [
    {
        'type': 'function',
        'function': {
            'name': 'get_patient_data',
            'description': 'Retrieve complete clinical record for a patient by their ID.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'patient_id': {'type': 'string', 'description': 'Patient ID, e.g. P0001'}
                },
                'required': ['patient_id']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'flag_clinical_risk',
            'description': (
                'Evaluate a patient lab values, vitals, and visit history against '
                'clinical thresholds. Returns risk level (LOW/MEDIUM/HIGH) and a list of flags.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'patient_id': {'type': 'string', 'description': 'Patient ID'}
                },
                'required': ['patient_id']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'generate_care_actions',
            'description': (
                'Generate a prioritised list of clinical follow-up care actions '
                'for a patient, based on their risk assessment output.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'patient_id': {'type': 'string', 'description': 'Patient ID'},
                    'risk_assessment': {
                        'type': 'object',
                        'description': 'Output dict from flag_clinical_risk tool'
                    }
                },
                'required': ['patient_id', 'risk_assessment']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'get_missed_appointment_patients',
            'description': 'Return the complete list of patients who missed their last scheduled appointment.',
            'parameters': {'type': 'object', 'properties': {}, 'required': []}
        }
    },
]

# Tool dispatcher — maps LLM function names to Python implementations
TOOL_MAP = {
    'get_patient_data':               lambda a: get_patient_data(**a),
    'flag_clinical_risk':             lambda a: flag_clinical_risk(**a),
    'generate_care_actions':          lambda a: generate_care_actions(**a),
    'get_missed_appointment_patients': lambda a: get_missed_appointment_patients(),
}

print('✅ Tool schemas registered:', [t['function']['name'] for t in TOOLS])

---
## Task 3 — Implement the Agentic Loop

The `run_agent()` function implements a **ReAct-style agentic loop**:
1. The LLM receives a clinical prompt and a list of available tools
2. It autonomously decides which tool to call (no hardcoded routing)
3. Tool results are fed back as `tool` messages
4. The LLM continues calling tools until it has enough information
5. It produces a final narrative response for the care team

```
User Prompt
    │
    ▼
┌─────────────────────────────┐
│  LLM (GPT-4o-mini)          │
│  Decides: call tool?        │
└────────────┬────────────────┘
             │ tool_call
             ▼
┌─────────────────────────────┐
│  Tool Dispatcher            │
│  get_patient_data           │
│  flag_clinical_risk         │  ← loop until no more tool calls
│  generate_care_actions      │
│  get_missed_appointments    │
└────────────┬────────────────┘
             │ tool result
             ▼
┌─────────────────────────────┐
│  LLM generates final answer │
└─────────────────────────────┘
```

In [ ]:
def run_agent(user_prompt: str, verbose: bool = True) -> str:
    """
    Agentic loop: LLM autonomously orchestrates tool calls to analyse patient
    data, flag risks, and generate care actions. Loops until no more tool calls.

    Args:
        user_prompt: Clinical task description for the agent
        verbose:     Print step-by-step tool call trace

    Returns:
        Final narrative response from the LLM
    """
    SYSTEM_PROMPT = (
        'You are MediCare Follow-Up Agent, a clinical AI assistant for a chronic disease '
        'outpatient clinic. Your role is to autonomously review patient records, identify '
        'clinical risks, and generate prioritised care actions for the coordination team.\n\n'
        'Guidelines:\n'
        '- Always call tools to gather data before drawing conclusions\n'
        '- Chain tools logically: get data → assess risk → generate actions\n'
        '- Present findings with: Patient name/ID, Risk Level, Key Flags, Recommended Actions\n'
        '- Use clinical language appropriate for a care coordination team\n'
        '- For HIGH risk patients, always emphasise urgency of contact'
    )

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': user_prompt}
    ]

    if verbose:
        print(f'\n🤖 Agent starting')
        print(f'📋 Prompt: "{user_prompt[:100]}"')
        print('─' * 65)

    for step in range(12):   # safety cap: max 12 tool-call rounds
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice='auto'
        )
        msg = response.choices[0].message
        messages.append(msg)

        # No tool calls → LLM has finished reasoning, return final answer
        if not msg.tool_calls:
            if verbose:
                print(f'\n✅ Agent completed in {step + 1} reasoning step(s).')
            return msg.content

        # Execute each tool call and append results
        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            if verbose:
                print(f'  🔧 Step {step+1}: {fn_name}({fn_args})')
            result = TOOL_MAP[fn_name](fn_args)
            messages.append({
                'role':         'tool',
                'tool_call_id': tc.id,
                'content':      json.dumps(result, default=str)
            })

    return '⚠️  Agent reached maximum steps without producing a final answer.'


print('✅ Agentic loop (run_agent) ready.')

---
## Task 4 — Single Patient Analysis

Run the agent on a single high-risk patient. The agent autonomously chains:
`get_patient_data` → `flag_clinical_risk` → `generate_care_actions`

**Patient P0008 — Deepa Krishnan** (Heart Failure, BNP=1100.1 pg/mL, dizziness & fatigue)

In [ ]:
# Task 4a: High-risk cardiac patient
result_p0008 = run_agent(
    'Perform a complete clinical risk assessment for patient P0008. '
    'Identify all risk flags and generate a prioritised follow-up care plan.',
    verbose=True
)
print('\n' + '='*65)
print('AGENT RESPONSE:')
print('='*65)
print(result_p0008)

In [ ]:
# Task 4b: Critical mental health patient — PHQ-9 = 26.6/27
result_p0028 = run_agent(
    'Assess patient P0028 for clinical risks and generate urgent care actions.',
    verbose=True
)
print('\n' + '='*65)
print('AGENT RESPONSE:')
print('='*65)
print(result_p0028)

In [ ]:
# Task 4c: Complex multi-morbidity patient (COPD + Heart Failure)
result_p0074 = run_agent(
    'Review patient P0074 who has multiple diagnoses. '
    'Flag all clinical risks and create a coordinated care action plan.',
    verbose=True
)
print('\n' + '='*65)
print('AGENT RESPONSE:')
print('='*65)
print(result_p0074)

---
## Task 5 — Missed Appointment Follow-Up

The agent autonomously:
1. Calls `get_missed_appointment_patients` to retrieve all patients who missed appointments
2. Calls `flag_clinical_risk` for each patient to assess urgency
3. Produces a **prioritised outreach list** sorted by risk level (HIGH → MEDIUM → LOW)

In [ ]:
# Task 5: Missed appointment batch follow-up
result_missed = run_agent(
    'Identify all patients who missed their last appointment. '
    'For each patient, assess their clinical risk level. '
    'Present a prioritised outreach list: HIGH risk first, then MEDIUM, then LOW. '
    'Include patient name, diagnosis, risk level, and top 2 care actions for each.',
    verbose=True
)
print('\n' + '='*65)
print('PRIORITISED MISSED APPOINTMENT OUTREACH LIST:')
print('='*65)
print(result_missed)

---
## Bonus — Batch Risk Dashboard (All 100 Patients)

Run `flag_clinical_risk` for all patients programmatically and visualise the risk distribution.

In [ ]:
# Batch-process all 100 patients
batch_results = [flag_clinical_risk(pid) for pid in df['patient_id']]
risk_df = pd.DataFrame(batch_results)

print('Risk Level Distribution:')
print(risk_df['risk_level'].value_counts().to_string())
print(f'\nAverage flags per patient: {risk_df["flag_count"].mean():.2f}')
risk_df.head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Patient Risk Assessment Dashboard — All 100 Patients', fontsize=13, fontweight='bold')

RISK_COLORS = {'HIGH': '#e74c3c', 'MEDIUM': '#f39c12', 'LOW': '#2ecc71'}

# 1. Risk level pie chart
counts = risk_df['risk_level'].value_counts()
axes[0].pie(
    counts,
    labels=counts.index,
    autopct='%1.0f%%',
    colors=[RISK_COLORS[c] for c in counts.index],
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title('Risk Level Distribution')

# 2. Flag count histogram
axes[1].hist(
    risk_df['flag_count'],
    bins=range(0, risk_df['flag_count'].max() + 2),
    color='#3498db', edgecolor='white', align='left'
)
axes[1].set_title('Clinical Flags per Patient')
axes[1].set_xlabel('Number of Flags')
axes[1].set_ylabel('Number of Patients')

# 3. Risk level by diagnosis (top 8 diagnoses)
top_diag = df['diagnosis'].value_counts().head(8).index
risk_by_diag = risk_df[risk_df['diagnosis'].isin(top_diag)].groupby(
    ['diagnosis', 'risk_level']).size().unstack(fill_value=0)
risk_by_diag = risk_by_diag.reindex(columns=['HIGH','MEDIUM','LOW'], fill_value=0)
risk_by_diag.plot(
    kind='barh', ax=axes[2], stacked=True,
    color=[RISK_COLORS[c] for c in ['HIGH','MEDIUM','LOW']]
)
axes[2].set_title('Risk by Diagnosis (Top 8)')
axes[2].invert_yaxis()
axes[2].legend(title='Risk Level', bbox_to_anchor=(1.02, 1))

plt.tight_layout()
plt.savefig('risk_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Print Top 15 highest-risk patients
risk_priority = {'HIGH': 0, 'MEDIUM': 1, 'LOW': 2}
top15 = risk_df.copy()
top15['priority'] = top15['risk_level'].map(risk_priority)
top15 = top15.sort_values(['priority', 'flag_count'], ascending=[True, False]).head(15)

print('\n🚨 TOP 15 HIGHEST-RISK PATIENTS — Priority Outreach List')
print('='*100)
print(f'{"#":<3} {"ID":<7} {"Name":<25} {"Age":<5} {"Risk":<8} {"Flags":<6} {"Diagnosis"}')
print('-'*100)
for i, (_, row) in enumerate(top15.iterrows(), 1):
    emoji = '🔴' if row['risk_level']=='HIGH' else ('🟡' if row['risk_level']=='MEDIUM' else '🟢')
    print(f"{i:<3} {row['patient_id']:<7} {row['patient_name']:<25} {int(row['age']):<5} "
          f"{emoji}{row['risk_level']:<7} {row['flag_count']:<6} {str(row['diagnosis'])[:45]}")

---
## Summary

### Architecture

| Component | Details |
|-----------|--------|
| **LLM** | OpenAI GPT-4o-mini via function-calling API |
| **Agentic Pattern** | ReAct-style autonomous tool-chaining loop |
| **Decision Making** | LLM decides which tools to call and in what order — no hardcoded routing |
| **Tool Execution** | Python tool dispatcher maps LLM function names to implementations |
| **Safety** | Maximum 12 reasoning steps to prevent infinite loops |

### Tools
| Tool | Purpose | Returns |
|------|---------|--------|
| `get_patient_data` | Fetch full record | Patient dict |
| `flag_clinical_risk` | Threshold evaluation | Risk level + flags |
| `generate_care_actions` | Action planning | Prioritised action list |
| `get_missed_appointment_patients` | Batch query | Patient list |

### Risk Stratification
- 🔴 **HIGH** — 3 or more clinical flags → urgent contact within 24 hours
- 🟡 **MEDIUM** — 1–2 flags → priority review within 1 week  
- 🟢 **LOW** — 0 flags → routine follow-up at next scheduled visit

### Key Findings from Batch Analysis
The agent identified patients at highest risk including those with:
- Severely elevated BNP (>1000 pg/mL) in Heart Failure patients
- Critical PHQ-9 scores near maximum (26.6/27)
- FEV1% <30% indicating very severe COPD
- Multiple compounding risk factors (missed appointments + abnormal labs + elevated vitals)

> **Note:** This is a prototype. Clinical decisions must always be validated by qualified healthcare professionals.